# 🧩 Visual Rebus Puzzle Decoding — Competition Solution

**Architecture:** Qwen2.5-VL-7B-Instruct + QLoRA Fine-tuning + Multi-candidate Voting


**Metric:** Exact Match Accuracy (case-insensitive, whitespace-stripped)

---

### 🔧 `LOCAL_TEST` Mode

Set `LOCAL_TEST = True` in Step 1 to run a **quick smoke test** using minimal data:

| Setting | `LOCAL_TEST = True` | `LOCAL_TEST = False` |
|---------|--------------------|-----------------------|
| Training samples | 5 | All 541 |
| Training epochs | 1 | 3 |
| Gradient accum | 2 | 8 |
| Max seq length | 1024 | 2048 |
| Image pixel budget | 384×28² | 512×28² |
| Test inference | 3 samples | All 90 |
| Multi-candidate voting | Off (greedy only) | On (beam + 3 temps) |
| Validation check | 2 samples | 20 samples |
| Adapter save | Skipped | Saved |
| Checkpoint save | Skipped | Per epoch |

Flip to `False` for the real submission run.

Dataset: https://www.kaggle.com/datasets/naveenvenubagadi/vrebus

## Step 0: Install Dependencies

In [ ]:
%%capture
!pip install -q transformers>=4.46.0 accelerate>=0.34.0 bitsandbytes>=0.44.0
!pip install -q peft>=0.13.0 trl>=0.12.0 qwen-vl-utils
!pip install -q datasets rapidfuzz pillow pandas numpy scikit-learn
!pip install -q jellyfish  # phonetic hashing: Metaphone, Soundex, NYSIIS


## Step 1: Configuration & Imports

**⬇️ Flip this single switch to control everything ⬇️**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔀  THE SWITCH                                                ║
# ║                                                                 ║
# ║    True  →  quick smoke test  (5 train, 3 test, ~5 min)       ║
# ║    False →  full submission   (541 train, 90 test, ~2-3 hrs)  ║
# ╚══════════════════════════════════════════════════════════════════╝

LOCAL_TEST = False   # ← CHANGE TO False FOR REAL SUBMISSION

# ──────────────────────────────────────────────────────────────────

import os, re, json, logging, warnings, math, gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
from PIL import Image


warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')
log = logging.getLogger('rebus')

# ── Environment Detection ──
ON_KAGGLE = os.path.exists('/kaggle/input')
BASE = '/kaggle/input/datasets/naveenvenubagadi/vrebus' if ON_KAGGLE else './dataset/public'
OUT  = '/kaggle/working' if ON_KAGGLE else './working'
os.makedirs(OUT, exist_ok=True)

TRAIN_CSV    = f'{BASE}/train.csv'
TEST_CSV     = f'{BASE}/test.csv'
TRAIN_IMAGES = f'{BASE}/train_images'
TEST_IMAGES  = f'{BASE}/test_images'
SUBMISSION   = f'{OUT}/submission.csv'

# ── GPU Detection ──
if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    IS_A10G = VRAM >= 20
    log.info(f'GPU: {GPU} | VRAM: {VRAM:.1f} GB | A10G={IS_A10G}')
else:
    IS_A10G = False
    log.warning('No GPU detected!')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Everything below is derived from LOCAL_TEST — no other changes
# needed anywhere in the notebook.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MODEL_ID       = 'Qwen/Qwen2.5-VL-7B-Instruct'
LORA_RANK      = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0.05
LEARNING_RATE  = 2e-4
WARMUP_RATIO   = 0.1

if LOCAL_TEST:
    NUM_EPOCHS     = 1
    GRAD_ACCUM     = 2
    MAX_SEQ_LEN    = 1024
    MAX_NEW_TOKENS = 64
    TRAIN_LIMIT    = 5       # only 5 training images
    TEST_LIMIT     = 3       # only 3 test images
    VAL_SAMPLES    = 2       # only 2 validation checks
    USE_VOTING     = False   # greedy decode only
    NUM_BEAMS      = 1       # no beam search
    LOG_EVERY      = 1       # log every step
    PIXEL_BUDGET   = 384     # smaller images
else:
    NUM_EPOCHS     = 5       # more epochs for better convergence
    GRAD_ACCUM     = 4       # more frequent updates
    MAX_SEQ_LEN    = 2048
    MAX_NEW_TOKENS = 128
    TRAIN_LIMIT    = None    # use all 541
    TEST_LIMIT     = None    # use all 90
    VAL_SAMPLES    = 20
    USE_VOTING     = True    # beam + temperature sampling
    NUM_BEAMS      = 5
    LOG_EVERY      = 20
    PIXEL_BUDGET   = 512     # full resolution

MODE_TAG = 'LOCAL SMOKE TEST' if LOCAL_TEST else 'FULL SUBMISSION'
log.info(f'\n{"═"*60}')
log.info(f'MODE: {MODE_TAG}')
log.info(f'  Train samples : {TRAIN_LIMIT or "ALL (541)"}')
log.info(f'  Test samples  : {TEST_LIMIT or "ALL (90)"}')
log.info(f'  Epochs        : {NUM_EPOCHS}')
log.info(f'  Grad accum    : {GRAD_ACCUM}')
log.info(f'  Seq length    : {MAX_SEQ_LEN}')
log.info(f'  Pixel budget  : {PIXEL_BUDGET}×28²')
log.info(f'  Voting        : {USE_VOTING}')
log.info(f'  Beams         : {NUM_BEAMS}')
log.info(f'  Val samples   : {VAL_SAMPLES}')
log.info(f'{"═"*60}\n')


## Step 2: Load & Prepare Data

In [ ]:
train_df_full = pd.read_csv(TRAIN_CSV)
test_df_full  = pd.read_csv(TEST_CSV)

def normalize_label(s: str) -> str:
    """Normalize label for exact match evaluation.
    CRITICAL: Keep apostrophes! don't != dont in exact match.
    Only strip: trailing punct, parenthetical hints, double quotes.
    """
    s = str(s).lower().strip()
    s = re.sub(r'[.!?]+$', '', s)              # trailing .!? only
    s = re.sub(r'\s*\(.*?\)\s*', ' ', s)       # parenthetical hints
    s = re.sub(r'[\x22\x60]+', '', s)          # double quotes and backticks only
    s = re.sub(r'\s+', ' ', s).strip()
    return s

train_df_full['label_clean'] = train_df_full['label'].apply(normalize_label)

# Always keep the full vocabulary for fuzzy matching
KNOWN_PHRASES = set(train_df_full['label_clean'].unique())

# Curriculum complexity scoring
def complexity(label):
    w = label.split()
    d = any(c.isdigit() for c in label)
    if len(w) == 1: return 0
    if len(w) <= 3 and not d: return 1
    if len(w) <= 5: return 2
    return 3

train_df_full['complexity'] = train_df_full['label_clean'].apply(complexity)
train_df_sorted = train_df_full.sort_values('complexity').reset_index(drop=True)

# Apply limits based on LOCAL_TEST
if TRAIN_LIMIT is not None:
    train_df_use = train_df_sorted.head(TRAIN_LIMIT).copy()
else:
    train_df_use = train_df_sorted.copy()

if TEST_LIMIT is not None:
    test_df_use = test_df_full.head(TEST_LIMIT).copy()
else:
    test_df_use = test_df_full.copy()

log.info(f'Data ready - train: {len(train_df_use)}, test: {len(test_df_use)}, vocab: {len(KNOWN_PHRASES)}')
if LOCAL_TEST:
    print('\nTraining subset:')
    print(train_df_use[['image_id', 'label_clean', 'complexity']].to_string(index=False))
    print('\nTest subset:')
    print(test_df_use[['id', 'image_id']].to_string(index=False))


## Step 3: Load Qwen2.5-VL-7B with 4-bit Quantization

In [ ]:
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import importlib

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if IS_A10G else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Detect flash_attn package directly
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
ATTN_IMPL = "flash_attention_2" if HAS_FLASH_ATTN else "sdpa"
log.info(f'Flash Attention: {HAS_FLASH_ATTN} -> {ATTN_IMPL}')

log.info(f'Loading {MODEL_ID}...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if IS_A10G else torch.float16,
    attn_implementation=ATTN_IMPL,
    trust_remote_code=True,
)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = PIXEL_BUDGET * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

log.info(f'Model loaded! pixel_budget={PIXEL_BUDGET}x28sq={MAX_PIXELS:,}px')


## Step 4: Apply QLoRA Adapters

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
log.info(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Step 5: Build Training Dataset

In [ ]:
from datasets import Dataset
from qwen_vl_utils import process_vision_info
import re as _re

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# KEY INSIGHT FROM RESEARCH:
# The "Say it fast" auditory blending heuristic from phonics education.
# Instead of analyzing meaning, concatenate elements and SOUND THEM OUT.
# CAT + A + COMB → say fast → "catacomb"
# LEM on ADE → say fast → "lemonade"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SYSTEM_PROMPT = '''You are an expert rebus puzzle decoder. Follow these steps IN ORDER:

STEP 1 - VISUAL INVENTORY: List ALL text, numbers, symbols, objects, and images you see. Note font sizes, colors, positions, and visual effects (broken, crossed out, rotated, etc).

STEP 2 - SPATIAL MAPPING: Describe spatial relationships:
  - Text ABOVE/OVER other text -> insert the word "over" between them
  - Text BELOW/UNDER other text -> insert "under"
  - Text INSIDE other text -> insert "in"
  - Text BESIDE other text -> note adjacency
  - BIG text = "big", SMALL text = "little/small"
  - Text going DOWN = "down" prefix
  - BROKEN text = "broken", CROSSED OUT = "corrected" or "not"
  - Repeated text N times = the number N (4 = "for", 2 = "to/two", 8 = "ate")

STEP 3 - SYMBOL TRANSLATION: Convert visual symbols to words:
  - Chess knight piece = "night"
  - Female symbol = "miss"
  - Prohibition circle = "don\'t" or "no"
  - Match/fire = "match"
  - Barrier/sawhorse = "end"
  - Number 4 = "for", 2 = "to/two", 8 = "ate"

STEP 4 - "SAY IT FAST" PHONETIC FUSION:
  Take ALL the words/syllables from steps 2-3 and CONCATENATE them.
  Remove spaces and SAY THE RESULT FAST out loud.
  Does it sound like a real English word or common phrase?
  Examples:
    CAT + A + COMB -> "catacomb"
    LEM over ADE -> LEM + ON + ADE -> "lemonade"
    MISS + MATCH -> "mismatch"
    S + CAPE -> "escape"
    DE + FENCE -> "defense"
    CAR + EARS -> "careers"
    BREAK + FAST -> "breakfast"
    camping OVER night -> "camping overnight"
    BIG + bad + wolf -> "big bad wolf"

STEP 5 - ANSWER: Write the decoded English phrase.
Keep apostrophes (don\'t, he\'s, it\'s, i\'m, you\'re, i\'ll).

Output your step-by-step reasoning, then on the LAST line write:
Answer: [phrase in lowercase]'''

def generate_cot_label(label_raw):
    """Auto-generate chain-of-thought from the known answer."""
    label = label_raw.lower().strip()
    label = _re.sub(r'[.!?]+$', '', label)
    label = _re.sub(r'\s*\(.*?\)\s*', ' ', label)
    label = _re.sub(r'[\x22\x60]+', '', label)
    label = _re.sub(r'\s+', ' ', label).strip()
    
    steps = []
    
    # SIZE
    if label.startswith('big '):
        steps.append(f'Text in LARGE font = "big". Content: "{label[4:]}".')
    if label.startswith('little ') or label.startswith('small '):
        steps.append('Text in TINY font = "little" or "small".')
    if label.startswith('long '):
        steps.append('Text stretched/elongated = "long".')
    if label.startswith('short '):
        steps.append('Text compressed = "short".')
    
    # OVER compounds
    for ow in ['overnight','overboard','overcome','overlook','overpass','overtime',
                'overdue','overweight','overflow','overhead']:
        if ow in label:
            root = ow[4:]
            steps.append(f'Text ABOVE "{root}". Position=over. Say it fast: over+{root} = {ow}.')
    if ' over ' in label:
        steps.append('One element ABOVE another = "over".')
    if 'top secret' in label:
        steps.append('"SECRET" at the TOP = "top secret".')
    elif label.startswith('top '):
        steps.append('Text at TOP of image/element.')
    
    # UNDER
    if 'understand' in label:
        steps.append('"STAND" with I UNDER it. Say it fast: under+stand = understand.')
    elif ' under ' in label:
        steps.append('Element UNDER/BELOW another = "under".')
    
    # INSIDE/BETWEEN
    if 'inside' in label: steps.append('Elements INSIDE another.')
    if 'between' in label: steps.append('Element BETWEEN two others.')
    if 'beside' in label: steps.append('Elements BESIDE each other.')
    
    # COMPOUND phonetic fusions (the 46% gap!)
    compound_map = {
        'catacomb': 'CAT + A + COMB. Say it fast: catacomb.',
        'careers': 'CAR + EARS. Say it fast: careers.',
        'escape': 'S + CAPE. Say it fast: escape.',
        'mismatch': 'MISS (female symbol) + MATCH (fire). Say it fast: mismatch.',
        'defense': 'DE + FENCE. Say it fast: defense.',
        'lemonade': 'LEM on ADE. LEM+ON+ADE. Say it fast: lemonade.',
        'cashew': 'CASH + EW. Say it fast: cashew.',
        'canoe': 'CAN + OE. Say it fast: canoe.',
        'romantic': 'ROMAN + TIC. Say it fast: romantic.',
        'please': 'P + LEASE. Say it fast: please.',
        'blanket': 'BLANK + ET. Say it fast: blanket.',
        'rainbow': 'RAIN + BOW. Say it fast: rainbow.',
        'mousetrap': 'MOUSE + TRAP. Say it fast: mousetrap.',
        'seahorse': 'SEA + HORSE. Say it fast: seahorse.',
        'breakfast': 'BREAK + FAST. Say it fast: breakfast.',
        'downtown': 'Text going DOWN + TOWN. Say it fast: downtown.',
        'waterfall': 'WATER falling down. Say it fast: waterfall.',
        'potatoes': 'POT + 8 Os. POT+EIGHT+O = potatoes.',
        'comfortable': 'COME + 4 + TABLE. Say it fast: comfortable.',
        'horsefly': 'HORSE + FLY. Say it fast: horsefly.',
        'doghouse': 'DOG + HOUSE. Say it fast: doghouse.',
        'bookcase': 'BOOK + CASE. Say it fast: bookcase.',
        'toothpick': 'TOOTH + PICK. Say it fast: toothpick.',
        'crossbow': 'CROSS + BOW. Say it fast: crossbow.',
        'crossroads': 'CROSS + ROADS. Say it fast: crossroads.',
        'stepfather': 'STEP + FATHER. Say it fast: stepfather.',
        'eyeshadow': 'EYE + SHADOW. Say it fast: eyeshadow.',
        'sandbox': 'S AND inside BOX. Say it fast: sandbox.',
        'tombstone': 'TOMB + STONE. Say it fast: tombstone.',
        'tricycle': 'TRI + CYCLE. Say it fast: tricycle.',
    }
    for word, cot in compound_map.items():
        if word in label.replace(' ', ''):
            steps.append(cot)
    
    # DIRECTION
    if label.startswith('down') and len(label) > 4 and not label.startswith('down '):
        if not any(s for s in steps if 'down' in s.lower()):
            steps.append('Text oriented DOWNWARD = "down" prefix.')
    if label.startswith('back') and len(label) > 4:
        steps.append('Text BACKWARDS or at the back.')
    
    # VISUAL STATE
    if 'broken' in label: steps.append('Text CRACKED/FRACTURED = "broken".')
    if 'split' in label: steps.append('Text SPLIT into parts = "split".')
    if 'missing' in label: steps.append('Part of text MISSING.')
    if 'scrambled' in label: steps.append('Letters SCRAMBLED.')
    if 'corrected' in label: steps.append('STRIKETHROUGH with correction = "corrected".')
    if 'divided' in label: steps.append('Text has GAP/DIVISION = "divided".')
    
    # SYMBOLS/NEGATION
    if "don't" in label or label.startswith('no '):
        steps.append('PROHIBITION symbol = "don\'t" or "no".')
    
    # COUNTING
    if ' for ' in label or label.startswith('for '):
        steps.append('Elements repeat 4 times (4 sounds like "for").')
    if 'two' in label or '2' in label:
        steps.append('Two elements or number 2 present.')
    
    # COLORS
    for c in ['red','blue','green','black','white','brown','purple']:
        if c in label.split():
            steps.append(f'Text in {c} color = "{c}".')
    
    # OBJECTS
    if 'night' in label: steps.append('Chess KNIGHT = "night".')
    if 'weekend' in label: steps.append('WEEK + barrier/END = weekend.')
    
    if not steps:
        steps.append('I observe text with specific spatial positions, sizes, and visual effects.')
    
    return ' '.join(steps) + '\nAnswer: ' + label

def build_messages(image_path, label=None):
    """Build Qwen2.5-VL chat messages."""
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': f'file://{image_path}'},
            {'type': 'text', 'text': 'Decode this rebus puzzle. Follow all 5 steps. End with Answer: [phrase]'},
        ]},
    ]
    if label is not None:
        msgs.append({'role': 'assistant', 'content': generate_cot_label(label)})
    return msgs

# Build flat dataset
train_records = []
skipped = 0
for _, row in train_df_use.iterrows():
    img_path = os.path.join(TRAIN_IMAGES, row['image_id'])
    if not os.path.exists(img_path):
        skipped += 1
        continue
    train_records.append({
        'image_path': img_path,
        'label': row['label_clean'],
    })

train_dataset = Dataset.from_list(train_records)
log.info(f'Training dataset: {len(train_dataset)} examples (skipped {skipped} missing)')

if LOCAL_TEST:
    for rec in train_records[:2]:
        print(f'\nLabel: {rec["label"]}')
        print(f'CoT: {generate_cot_label(rec["label"])}')


## Step 6: Custom Multi-Modal Collator

In [ ]:
class QwenVLCollator:
    """Handles image processing, chat template, and label masking.
    Rebuilds messages from flat image_path + label fields.
    """

    def __init__(self, processor, max_length=2048):
        self.processor = processor
        self.max_length = max_length
        self.tokenizer = processor.tokenizer

    def __call__(self, examples):
        texts, all_images = [], []

        for ex in examples:
            # Rebuild messages fresh from flat fields
            msgs = build_messages(ex['image_path'], ex['label'])

            text = self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images, _ = process_vision_info(msgs)
            if images:
                all_images.extend(images)

        batch = self.processor(
            text=texts,
            images=all_images if all_images else None,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt',
        )

        # Label masking: only compute loss on assistant response tokens
        labels = batch['input_ids'].clone()
        assistant_marker = self.tokenizer.encode(
            '<|im_start|>assistant\n', add_special_tokens=False
        )
        marker_len = len(assistant_marker)

        for i in range(len(labels)):
            ids = labels[i]
            found = -1
            for j in range(len(ids) - marker_len):
                if ids[j:j+marker_len].tolist() == assistant_marker:
                    found = j + marker_len
            if found > 0:
                labels[i, :found] = -100
            labels[i][ids == self.tokenizer.pad_token_id] = -100

        batch['labels'] = labels
        return batch

collator = QwenVLCollator(processor, max_length=MAX_SEQ_LEN)


## Step 7: QLoRA Fine-Tuning

In [ ]:
training_args = TrainingArguments(
    output_dir=f'{OUT}/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=IS_A10G,
    fp16=not IS_A10G and torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=LOG_EVERY,
    logging_first_step=True,
    save_strategy='no' if LOCAL_TEST else 'epoch',
    save_total_limit=2,
    dataloader_num_workers=0 if LOCAL_TEST else 2,
    dataloader_pin_memory=not LOCAL_TEST,
    remove_unused_columns=False,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

log.info(f'Fine-tuning: {len(train_dataset)} samples × {NUM_EPOCHS} epoch(s), '
         f'grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}')
trainer.train()
log.info('Fine-tuning complete!')

if not LOCAL_TEST:
    ADAPTER_PATH = f'{OUT}/rebus_lora'
    model.save_pretrained(ADAPTER_PATH)
    processor.save_pretrained(ADAPTER_PATH)
    log.info(f'Adapter saved: {ADAPTER_PATH}')
else:
    log.info('⚡ LOCAL_TEST: skipping adapter save')

## Step 8: Post-Processing

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PHONETIC POST-PROCESSING PIPELINE
# From research: use Metaphone hashing + "say it fast" concat
# to bridge the gap between VLM output and exact match answers
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Pre-build phonetic lookup table from known phrases
try:
    import jellyfish
    HAS_JELLYFISH = True
    log.info('jellyfish loaded - phonetic matching enabled')
except ImportError:
    HAS_JELLYFISH = False
    log.warning('jellyfish not available - phonetic matching disabled')

# Build phonetic index of all known phrases
PHONETIC_INDEX = {}  # metaphone_hash -> list of phrases
if HAS_JELLYFISH:
    for phrase in KNOWN_PHRASES:
        # Hash the concatenated (no spaces) version
        concat = phrase.replace(' ', '').replace('-', '').replace("'", '')
        try:
            mph = jellyfish.metaphone(concat)
            if mph not in PHONETIC_INDEX:
                PHONETIC_INDEX[mph] = []
            PHONETIC_INDEX[mph].append(phrase)
        except:
            pass
    log.info(f'Phonetic index: {len(PHONETIC_INDEX)} unique hashes for {len(KNOWN_PHRASES)} phrases')

def say_it_fast(text):
    """Apply the "say it fast" heuristic from the research paper.
    Concatenate tokens, expand numbers, try to match known words.
    """
    text = text.lower().strip()
    
    # Number-to-phonetic expansion
    num_map = {'4': 'for', '8': 'eight', '2': 'to', '1': 'one',
               '3': 'three', '6': 'six', '10': 'ten'}
    expanded = text
    for num, word in sorted(num_map.items(), key=lambda x: -len(x[0])):
        expanded = expanded.replace(num, word)
    
    # Concatenate (remove spaces) = "say it fast"
    concat = expanded.replace(' ', '').replace('-', '')
    
    # Check 1: Direct match against known phrases (no spaces)
    for phrase in KNOWN_PHRASES:
        phrase_concat = phrase.replace(' ', '').replace('-', '').replace("'", '')
        if concat == phrase_concat:
            return phrase
    
    # Check 2: Phonetic hash match via Metaphone
    if HAS_JELLYFISH:
        try:
            candidate_hash = jellyfish.metaphone(concat)
            if candidate_hash in PHONETIC_INDEX:
                matches = PHONETIC_INDEX[candidate_hash]
                if len(matches) == 1:
                    return matches[0]
                # Multiple matches: pick closest by edit distance
                best = min(matches, key=lambda m: jellyfish.levenshtein_distance(concat, m.replace(' ', '')))
                return best
        except:
            pass
    
    return None  # No phonetic match found

def extract_answer_from_cot(raw):
    """Extract the final answer from chain-of-thought output."""
    text = raw.strip()
    matches = list(re.finditer(r'[Aa]nswer:\s*(.+?)$', text, re.MULTILINE))
    if matches:
        return matches[-1].group(1).strip()
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if lines:
        return lines[-1]
    return text

def post_process(raw, known=KNOWN_PHRASES):
    """Full post-processing: CoT extraction + phonetic fusion + fuzzy match."""
    # Step 1: Extract answer from CoT reasoning
    pred = extract_answer_from_cot(raw)
    pred = pred.lower().strip()

    # Step 2: Strip model preamble
    prev = None
    while prev != pred:
        prev = pred
        for pfx in ['the answer is ', 'answer: ', 'the phrase is ',
                     'decoded: ', 'solution: ', 'this represents ']:
            if pred.startswith(pfx):
                pred = pred[len(pfx):]

    # Step 3: Strip wrapping quotes
    if len(pred) >= 2 and pred[0] == pred[-1] and pred[0] in '"\x60':
        pred = pred[1:-1]
    pred = pred.strip('"\x60*_')
    pred = re.sub(r'[.!?,;:]+$', '', pred)
    pred = re.sub(r'\s*\(.*?\)\s*', ' ', pred)
    pred = re.sub(r'\s+', ' ', pred).strip()

    # Step 4: Try direct match first
    if pred in known:
        return pred

    # Step 5: "Say it fast" phonetic fusion
    phonetic_match = say_it_fast(pred)
    if phonetic_match:
        return phonetic_match

    # Step 6: Fuzzy string match at 90% threshold
    if known and pred:
        try:
            from rapidfuzz import fuzz, process as rfp
            match = rfp.extractOne(pred, list(known), scorer=fuzz.ratio)
            if match and match[1] >= 90:
                return match[0]
        except ImportError:
            pass

    return pred

# ── Test the pipeline ──
test_cases = [
    ('I see CAT, A, COMB. Say it fast: catacomb.\nAnswer: catacomb', 'catacomb'),
    ('BREAK + FAST\nAnswer: break fast', 'breakfast'),
    ('miss match', 'mismatch'),
    ('lemon ade', 'lemonade'),
    ('s cape', 'escape'),
    ("don't give up on me", "don't give up on me"),
    ('de fence', 'defense'),
    ('camp over night\nAnswer: camping overnight', 'camping overnight'),
]
print('Post-processing tests:')
all_ok = True
for inp, expected in test_cases:
    result = post_process(inp)
    ok = result == expected
    all_ok = all_ok and ok
    print(f'  {"Y" if ok else "N"} "{inp[:40]}..." -> "{result}" {"" if ok else f"(expected: {expected})"}')
if all_ok:
    print('All tests passed!')


## Step 9: Inference Function

In [ ]:
@torch.inference_mode()
def predict(image_path, model, processor, use_voting=USE_VOTING):
    "Generate prediction with chain-of-thought reasoning."
    msgs = build_messages(image_path)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    images, _ = process_vision_info(msgs)
    inputs = processor(
        text=[text], images=images, padding=True, return_tensors='pt'
    ).to(model.device)

    candidates = []

    # More tokens for CoT reasoning chain + answer
    cot_tokens = MAX_NEW_TOKENS * 3

    # Primary: beam search
    out = model.generate(**inputs, max_new_tokens=cot_tokens,
                         num_beams=NUM_BEAMS, do_sample=False,
                         repetition_penalty=1.1)
    gen = out[0][inputs['input_ids'].shape[1]:]
    raw = processor.tokenizer.decode(gen, skip_special_tokens=True)
    candidates.append(post_process(raw))

    if use_voting:
        for temp in [0.2, 0.4, 0.6, 0.8, 1.0]:
            out = model.generate(**inputs, max_new_tokens=cot_tokens,
                                 do_sample=True, temperature=temp, top_p=0.9)
            gen = out[0][inputs['input_ids'].shape[1]:]
            raw = processor.tokenizer.decode(gen, skip_special_tokens=True)
            candidates.append(post_process(raw))
        return Counter(candidates).most_common(1)[0][0]

    return candidates[0]


## Step 10: Run Inference on Test Set

In [ ]:
model.eval()
predictions = {}

log.info(f'Inference: {len(test_df_use)} images | voting={USE_VOTING} | beams={NUM_BEAMS}')

for idx, row in test_df_use.iterrows():
    img_id = row['image_id']
    img_path = os.path.join(TEST_IMAGES, img_id)

    if not os.path.exists(img_path):
        log.warning(f'Missing: {img_path}')
        predictions[row['id']] = ''
        continue

    try:
        pred = predict(img_path, model, processor)
        predictions[row['id']] = pred
        log.info(f'  [{row["id"]:3d}] {img_id} → "{pred}"')
    except Exception as e:
        log.error(f'  [{row["id"]:3d}] {img_id} — ERROR: {e}')
        try:
            pred = predict(img_path, model, processor, use_voting=False)
            predictions[row['id']] = pred
        except:
            predictions[row['id']] = ''

log.info(f'Done: {len(predictions)} predictions')

## Step 11: Generate submission.csv

In [ ]:
if LOCAL_TEST:
    # In local mode: CSV only contains the rows we actually ran
    submission = pd.DataFrame([
        {'id': row['id'], 'label': predictions.get(row['id'], '')}
        for _, row in test_df_use.iterrows()
    ])
else:
    # Full mode: every test ID present (required for submission)
    submission = pd.DataFrame([
        {'id': tid, 'label': predictions.get(tid, '')}
        for tid in sorted(test_df_full['id'].tolist())
    ])

submission.to_csv(SUBMISSION, index=False)

print(f'\nSubmission saved: {SUBMISSION}')
print(f'Shape: {submission.shape}')
print(f'Non-empty: {(submission["label"] != "").sum()}/{len(submission)}')
print()
print(submission.to_string(index=False))

## Step 12: Validation Spot-Check

In [ ]:
model.eval()
correct, total = 0, 0

val_df = train_df_full.sample(min(VAL_SAMPLES, len(train_df_full)), random_state=42)
log.info(f'Validation spot-check: {len(val_df)} training samples')

for _, row in val_df.iterrows():
    img_path = os.path.join(TRAIN_IMAGES, row['image_id'])
    if not os.path.exists(img_path):
        continue
    try:
        pred = predict(img_path, model, processor, use_voting=False)
        truth = row['label_clean']
        match = pred.strip() == truth.strip()
        correct += int(match)
        total += 1
        status = '✓' if match else '✗'
        print(f'  {status}  pred="{pred}"  truth="{truth}"')
    except Exception as e:
        total += 1
        print(f'  ✗  ERROR on {row["image_id"]}: {e}')

acc = correct / max(total, 1)
print(f'\nValidation: {correct}/{total} = {acc:.1%} exact match')

if LOCAL_TEST:
    print()
    print('═' * 60)
    print('✅ LOCAL SMOKE TEST COMPLETE — pipeline runs end-to-end!')
    print()
    print('   Next steps:')
    print('   1. Set LOCAL_TEST = False in Step 1')
    print('   2. Re-run all cells')
    print('   3. Submit the generated submission.csv')
    print('═' * 60)